# Logistic Regression via Maximum Likelihood (From Scratch)

_Generated: 2025-10-22T17:36:33.484718Z_

**Objective.** Build **binary logistic regression** from first principles as a **maximum likelihood** estimator: derive the negative log-likelihood, its **gradient** and **Hessian**, and optimize with **gradient descent** (with backtracking line search) and **Newton/IRLS**. We include L2 regularization, convergence diagnostics, ROC/AUC and calibration plots, and an optional CSV upload.

**No ML libraries are used**—just NumPy/Matplotlib.

## 0) Environment & Dependencies

In [ ]:
import sys, platform, subprocess
def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)

!pip -q install --upgrade pip
!pip -q install numpy matplotlib

## 1) Imports, Reproducibility, and Plot Helpers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
SEED = 123
rng = np.random.default_rng(SEED)

def train_test_split(X, y, test_size=0.25, rng=rng):
    n = len(y); idx = np.arange(n); rng.shuffle(idx)
    n_test = int(np.round(test_size*n))
    te = idx[:n_test]; tr = idx[n_test:]
    return X[tr], X[te], y[tr], y[te]

def standardize(X, mean=None, std=None, eps=1e-12):
    if mean is None: mean = X.mean(axis=0)
    if std is None: std = X.std(axis=0)
    std = np.where(std < eps, 1.0, std)
    return (X - mean)/std, mean, std

def add_intercept(X):
    return np.hstack([np.ones((X.shape[0],1)), X])

def plot_decision_boundary(ax, W, mean, std, color=None, levels=[0.5]):
    # Only for 2D features X (after standardization). W includes intercept.
    if W is None or len(W) != 3: return
    x_min, x_max = -3.0, 3.0
    y_min, y_max = -3.0, 3.0
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    Xg = np.c_[xx.ravel(), yy.ravel()]
    Xg_raw = Xg*std + mean
    Xg_std, _, _ = standardize(Xg_raw, mean, std)
    Xg_phi = add_intercept(Xg_std)
    z = 1/(1+np.exp(-Xg_phi@W))
    Z = z.reshape(xx.shape)
    cs = ax.contour(xx, yy, Z, levels=levels)
    return cs

## 2) Data — Synthetic Sets and CSV Upload

We provide two synthetic datasets (linearly separable and mildly overlapping) and an optional **CSV upload**. For your CSV, set the `LABEL_COL` (0/1) and select `FEATURE_COLS` (numeric).

In [ ]:
# Synthetic data generators
def make_linear(n=400):
    # 2D, linearly separable-ish with some noise
    mean1 = np.array([0.5, 0.5]); mean0 = np.array([-0.5, -0.5])
    X1 = rng.normal(mean1, 0.6, size=(n//2, 2))
    X0 = rng.normal(mean0, 0.6, size=(n//2, 2))
    y = np.r_[np.ones(n//2), np.zeros(n//2)]
    X = np.vstack([X1, X0])
    idx = rng.permutation(n)
    return X[idx], y[idx]

def make_overlap(n=500):
    # Two Gaussians with stronger overlap
    mean1 = np.array([1.0, 0.0]); mean0 = np.array([0.0, 1.0])
    cov = np.array([[1.0, 0.6],[0.6, 1.0]])
    L = np.linalg.cholesky(cov)
    X1 = mean1 + rng.normal(size=(n//2, 2)) @ L.T
    X0 = mean0 + rng.normal(size=(n//2, 2)) @ L.T
    y = np.r_[np.ones(n//2), np.zeros(n//2)]
    X = np.vstack([X1, X0])
    idx = rng.permutation(n)
    return X[idx], y[idx]

# Pick dataset here:
DATASET = "overlap"   # "linear" or "overlap"
if DATASET == "linear":
    X, y = make_linear(400)
else:
    X, y = make_overlap(600)

# Optional: CSV upload (in Colab)
LABEL_COL = None          # e.g., "label" or integer index
FEATURE_COLS = None       # e.g., ["x1","x2","x3"] or [0,1,2]
USE_CSV = False

try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB and USE_CSV:
    print("Upload a CSV file...")
    up = files.upload()
    import io, csv
    name = list(up.keys())[0]
    raw = up[name].decode("utf-8", errors="ignore")
    # Very lightweight CSV reader (expects header)
    reader = csv.reader(io.StringIO(raw))
    rows = list(reader)
    header = rows[0]
    data = rows[1:]
    # Infer numeric columns
    numeric = []
    for c in range(len(header)):
        try:
            float(data[0][c]); numeric.append(c)
        except Exception:
            pass
    if LABEL_COL is None:
        print("Please set LABEL_COL to a column name or index containing 0/1 labels.")
    if FEATURE_COLS is None:
        print("Auto-selecting numeric columns as features (excluding label).")
        FEATURE_COLS = [c for c in numeric if (c != LABEL_COL and header[c] != LABEL_COL)]
    # Map labels to 0/1
    def as_num(x):
        try: return float(x)
        except: return np.nan
    M = np.array([[as_num(v) for v in row] for row in data], dtype=float)
    if isinstance(LABEL_COL, str):
        y = M[:, header.index(LABEL_COL)]
    else:
        y = M[:, LABEL_COL if LABEL_COL is not None else -1]
    X = M[:, FEATURE_COLS]
    # Attempt to coerce labels to {0,1}
    y = (y > np.nanmedian(y)).astype(float)

print("X shape:", X.shape, "| y mean:", y.mean())

## 3) Preprocess — Standardize and Train/Test Split

In [ ]:
X_std, mu, sigma = standardize(X)
X_tr, X_te, y_tr, y_te = train_test_split(X_std, y, test_size=0.25, rng=rng)
print("Train size:", X_tr.shape, "Test size:", X_te.shape)

# Add intercept
Phi_tr = add_intercept(X_tr)
Phi_te = add_intercept(X_te)

## 4) Logistic Regression: Likelihood, Gradient, Hessian

Model: For features $\phi(x) \in \mathbb{R}^{d+1}$ (intercept + standardized features), parameters $w\in\mathbb{R}^{d+1}$, probability $\Pr(y=1\mid x) = \sigma(w^\top\phi)$ where $\sigma(z)=1/(1+e^{-z})$.

For data $(\Phi, y)$, the **negative log-likelihood** (with L2 regularization $\lambda$ on non-intercept coefficients) is:
$$\mathcal{L}(w) = -\sum_i \big[y_i\log p_i + (1-y_i)\log(1-p_i)\big] + \tfrac{\lambda}{2}\lVert w_{1:}\rVert_2^2.$$
Its **gradient** and **Hessian** are:
$$\nabla\mathcal{L}(w) = \Phi^\top(p - y) + \lambda\,[0; w_{1:}]\,,\quad H = \Phi^\top W \Phi + \lambda\,\mathrm{diag}(0,1,\dots,1),$$
where $p=\sigma(\Phi w)$ and $W=\mathrm{diag}(p_i(1-p_i))$. We implement both **gradient descent with backtracking** and **Newton/IRLS** updates.

In [ ]:
def sigmoid(z):
    # stable sigmoid
    z = np.clip(z, -40, 40)
    return 1.0/(1.0 + np.exp(-z))

def nll_and_grad(Phi, y, w, lam=0.0):
    p = sigmoid(Phi @ w)
    eps = 1e-12
    nll = -np.sum(y*np.log(p+eps) + (1-y)*np.log(1-p+eps))
    # L2 on w[1:]
    nll += 0.5*lam*np.dot(w[1:], w[1:])
    grad = Phi.T @ (p - y)
    reg = np.r_[0.0, lam*w[1:]]
    return nll + 0.0, grad + reg, p

def hessian(Phi, p, lam=0.0):
    wdiag = p*(1-p)  # shape (n,)
    H = Phi.T @ (wdiag[:,None]*Phi)
    # L2 on non-intercept
    R = np.zeros_like(H)
    np.fill_diagonal(R, 0.0)
    for j in range(1, H.shape[0]):
        R[j,j] = lam
    return H + R

## 5) Optimizers — Backtracking Gradient Descent and Newton/IRLS

We implement two solvers. **GD** uses Armijo backtracking with a configurable step cap. **Newton/IRLS** uses the exact Hessian.
Both stop when the relative change in NLL is small or when reaching `max_iter`. We also clip steps if they explode.

In [ ]:
def fit_gd(Phi, y, lam=0.0, max_iter=500, tol=1e-6, alpha0=1.0, c=1e-4, rho=0.5, verbose=False):
    d = Phi.shape[1]
    w = np.zeros(d)
    history = []
    nll_prev = np.inf
    for it in range(1, max_iter+1):
        nll, g, p = nll_and_grad(Phi, y, w, lam)
        history.append(nll)
        if verbose and it % 50 == 0:
            print(f"[GD] it={it:03d} nll={nll:.6f}")
        if np.abs(nll_prev - nll) < tol*(1+nll):
            break
        # Backtracking line search (Armijo)
        alpha = alpha0
        dir = -g
        gTd = np.dot(g, dir)
        while True:
            w_new = w + alpha*dir
            nll_new, _, _ = nll_and_grad(Phi, y, w_new, lam)
            if nll_new <= nll + c*alpha*gTd or alpha < 1e-12:
                break
            alpha *= rho
        w = w_new
        nll_prev = nll
    return w, np.array(history)

def fit_newton(Phi, y, lam=0.0, max_iter=100, tol=1e-8, damping=1e-6, verbose=False):
    d = Phi.shape[1]
    w = np.zeros(d)
    history = []
    nll_prev = np.inf
    for it in range(1, max_iter+1):
        nll, g, p = nll_and_grad(Phi, y, w, lam)
        H = hessian(Phi, p, lam) + damping*np.eye(d)
        try:
            step = np.linalg.solve(H, g)
        except np.linalg.LinAlgError:
            step = np.linalg.pinv(H) @ g
        w_new = w - step
        nll_new, _, _ = nll_and_grad(Phi, y, w_new, lam)
        history.append(nll_new)
        if verbose:
            print(f"[NT] it={it:03d} nll={nll_new:.6f}")
        if np.abs(nll_prev - nll_new) < tol*(1+nll_new):
            w = w_new; break
        w = w_new; nll_prev = nll_new
    return w, np.array(history)

## 6) Train Models (GD and Newton) and Plot Convergence

In [ ]:
LAM = 1e-2  # L2 regularization strength

w_gd, hist_gd = fit_gd(Phi_tr, y_tr, lam=LAM, max_iter=600, tol=1e-7, alpha0=1.0, verbose=False)
w_nt, hist_nt = fit_newton(Phi_tr, y_tr, lam=LAM, max_iter=50, tol=1e-9, verbose=False)

print("GD weights:", w_gd)
print("NT weights:", w_nt)

fig = plt.figure(figsize=(6,4))
plt.plot(hist_gd, label="GD NLL")
plt.plot(hist_nt, label="Newton NLL")
plt.xlabel("Iteration"); plt.ylabel("Negative log-likelihood")
plt.title("Convergence")
plt.legend(); plt.tight_layout(); plt.show()

## 7) Evaluation — Accuracy, Precision/Recall, ROC & AUC (Manual), Calibration, Brier

In [ ]:
def predict_proba(Phi, w):
    return sigmoid(Phi @ w)

def predict_label(Phi, w, threshold=0.5):
    return (predict_proba(Phi, w) >= threshold).astype(float)

def confusion(y_true, y_pred):
    tp = np.sum((y_true==1) & (y_pred==1))
    tn = np.sum((y_true==0) & (y_pred==0))
    fp = np.sum((y_true==0) & (y_pred==1))
    fn = np.sum((y_true==1) & (y_pred==0))
    return int(tp), int(fp), int(tn), int(fn)

def precision_recall(tp, fp, tn, fn):
    prec = tp/(tp+fp+1e-12)
    rec  = tp/(tp+fn+1e-12)
    return prec, rec

def roc_curve_manual(y_true, scores):
    # thresholds from sorted unique scores
    idx = np.argsort(scores)[::-1]
    y = y_true[idx]; s = scores[idx]
    thresholds = np.r_[np.inf, s, -np.inf]
    tpr = []; fpr = []
    P = np.sum(y==1); N = np.sum(y==0)
    tp = fp = 0; last = np.inf
    for th in thresholds[1:]:
        # update tp/fp at each score (step-wise)
        mask = s >= th
        tp = int(np.sum(y[mask]==1))
        fp = int(np.sum(y[mask]==0))
        tpr.append(tp/(P+1e-12))
        fpr.append(fp/(N+1e-12))
    return np.array(fpr), np.array(tpr), thresholds[1:]

def auc_trapezoid(x, y):
    # assumes x increasing
    order = np.argsort(x)
    x = x[order]; y = y[order]
    return np.trapz(y, x)

def brier_score(y_true, p_hat):
    return np.mean((p_hat - y_true)**2)

# Evaluate both models
for name, w in [("GD", w_gd), ("Newton", w_nt)]:
    p_tr = predict_proba(Phi_tr, w); yhat_tr = (p_tr>=0.5).astype(float)
    p_te = predict_proba(Phi_te, w); yhat_te = (p_te>=0.5).astype(float)
    tp, fp, tn, fn = confusion(y_te, yhat_te)
    prec, rec = precision_recall(tp, fp, tn, fn)
    fpr, tpr, thr = roc_curve_manual(y_te, p_te)
    auc = auc_trapezoid(fpr, tpr)
    br = brier_score(y_te, p_te)
    print(f"[{name}] Test acc={np.mean(yhat_te==y_te):.3f}  precision={prec:.3f}  recall={rec:.3f}  AUC={auc:.3f}  Brier={br:.4f}")

# Plots: ROC and calibration
fig = plt.figure(figsize=(10,4))
ax1 = plt.subplot(1,2,1)
for name, w in [("GD", w_gd), ("Newton", w_nt)]:
    fpr, tpr, thr = roc_curve_manual(y_te, predict_proba(Phi_te, w))
    plt.plot(fpr, tpr, label=f"{name}")
plt.plot([0,1],[0,1], linestyle="--")
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC"); plt.legend()

ax2 = plt.subplot(1,2,2)
for name, w in [("GD", w_gd), ("Newton", w_nt)]:
    p = predict_proba(Phi_te, w)
    # reliability curve (binning)
    bins = np.linspace(0,1,11)
    idx = np.digitize(p, bins)-1
    bin_centers = 0.5*(bins[:-1]+bins[1:])
    rel = []
    for b in range(len(bin_centers)):
        mask = (idx==b)
        if np.any(mask):
            rel.append(np.mean(y_te[mask]))
        else:
            rel.append(np.nan)
    plt.plot(bin_centers, rel, marker='o', label=name)
plt.plot([0,1],[0,1], linestyle="--")
plt.xlabel("Predicted probability"); plt.ylabel("Empirical frequency")
plt.title("Calibration (Reliability)"); plt.legend()
plt.tight_layout(); plt.show()

## 8) 2D Visualization (if features are 2D)

In [ ]:
if X.shape[1] == 2:
    fig = plt.figure(figsize=(10,4))
    ax1 = plt.subplot(1,2,1)
    _ = ax1.scatter(X_tr[:,0], X_tr[:,1], c=y_tr, s=14)
    try:
        plot_decision_boundary(ax1, w_gd, mu, sigma)
    except Exception:
        pass
    ax1.set_title("GD decision boundary (std. space)"); ax1.set_xlabel("x1 (std)"); ax1.set_ylabel("x2 (std)")

    ax2 = plt.subplot(1,2,2)
    _ = ax2.scatter(X_tr[:,0], X_tr[:,1], c=y_tr, s=14)
    try:
        plot_decision_boundary(ax2, w_nt, mu, sigma)
    except Exception:
        pass
    ax2.set_title("Newton decision boundary (std. space)"); ax2.set_xlabel("x1 (std)"); ax2.set_ylabel("x2 (std)")
    plt.tight_layout(); plt.show()
else:
    print("Skipping 2D decision boundary plot (features are not 2D).")

## 9) Multiclass (One-vs-Rest) — Sketch

To extend to $K$ classes, train **K binary models** vs the rest using the same MLE objective; at prediction time choose the class with the **largest probability**. For strict probabilistic coherence you can also fit a **softmax regression** (multinomial logistic) using similar gradient/Hessian machinery (replace the sigmoid with softmax and remove the one-vs-rest reduction).

## 10) Save Artifacts & Download

We save learned weights, convergence histories, preprocessing stats, and test predictions. Use the helper below to download a ZIP.

In [ ]:
import os
os.makedirs("artifacts", exist_ok=True)

np.savez("artifacts/logreg_run.npz",
         w_gd=w_gd, w_nt=w_nt,
         hist_gd=hist_gd, hist_nt=hist_nt,
         mu=mu, sigma=sigma,
         X_te=X_te, y_te=y_te,
         p_te_gd=predict_proba(Phi_te, w_gd),
         p_te_nt=predict_proba(Phi_te, w_nt))

print("Artifacts:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        files.download('artifacts.zip')
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Colab download helper not available in this environment:', e)

## 11) Exercises & Extensions

- Add **elastic-net** regularization (L1+L2) via proximal gradient or coordinate descent.
- Implement **early stopping** and **cross-validation** for tuning `λ`.
- Compare solvers (GD vs Newton) on ill-conditioned data; add **preconditioning**.
- Extend to **multinomial logistic (softmax)** and compare with one-vs-rest on a 3-class synthetic dataset.